### Here we will use the journey stats table to cluster users

In [1]:
import gc
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
journey_path = BASE_DIR / "data/clean/journey_stats.csv"

In [3]:
journey_stats = pd.read_csv(journey_path)

In [5]:
journey_stats = journey_stats.drop(columns=["Unnamed: 0"])

In [7]:
journey_raw = journey_stats.copy()

In [6]:
journey_stats.head(10)

,journey_id,user_pseudo_id,tour_id,distinct_stories_touched,distinct_stories_completed,total_times_completed,max_story_completed_position,dropoff_position,total_stories,depth_ratio,...,sequential_transitions,sequential_ratio,completed_sequential_transitions,sequential_completed_ratio,distinct_sequential_transitions,distinct_completed_sequential_transitions,max_possible_transitions,global_sequential_coverage,global_completed_sequential_coverage,interactivity_pct
0,12146,c7f26129dd892cb7292f6283113726ad,107,67.0,11.0,12,167.0,220.0,220.0,0.050000,...,62.0,0.765432,8,0.098765,59.0,6.0,219.0,0.269406,0.027397,0.019704
1,840,0e1e29466bf2de31e00fe7a989673c99,822,2.0,1.0,1,2.0,2.0,82.0,0.012195,...,0.0,0.000000,0,0.000000,0.0,0.0,81.0,0.000000,0.000000,0.000000
2,841,0e1e29466bf2de31e00fe7a989673c99,822,48.0,23.0,28,81.0,82.0,82.0,0.280488,...,60.0,0.500000,33,0.275000,30.0,15.0,81.0,0.370370,0.185185,0.086364
3,7295,767fa15a9558b669f47d8e67e719918c,284,41.0,16.0,17,63.0,64.0,67.0,0.238806,...,14.0,0.127273,3,0.027273,13.0,2.0,66.0,0.196970,0.030303,0.076923
4,10605,ada97a097eabb072201591dd267767cd,447,4.0,0.0,0,0.0,32.0,73.0,0.000000,...,0.0,0.000000,0,0.000000,0.0,0.0,72.0,0.000000,0.000000,0.210526
5,10457,aa95a709d2859cb0b976835344d7df10,859,17.0,8.0,8,73.0,73.0,113.0,0.070796,...,3.0,0.150000,1,0.050000,3.0,1.0,112.0,0.026786,0.008929,0.015385
6,10842,b22d1794fe798eea9481aaaee845d122,240,39.0,26.0,26,77.0,79.0,80.0,0.325000,...,10.0,0.222222,5,0.111111,10.0,5.0,79.0,0.126582,0.063291,0.013029
7,10844,b22d1794fe798eea9481aaaee845d122,278,62.0,38.0,38,104.0,103.0,105.0,0.361905,...,9.0,0.107143,7,0.083333,9.0,7.0,104.0,0.086538,0.067308,0.004717
8,10845,b22d1794fe798eea9481aaaee845d122,862,11.0,9.0,9,40.0,40.0,42.0,0.214286,...,1.0,0.066667,1,0.066667,1.0,1.0,41.0,0.024390,0.024390,0.000000
9,9658,9cd63ad90a923a99be3d27a1e6085bd0,644,19.0,7.0,7,50.0,49.0,109.0,0.064220,...,7.0,0.233333,2,0.066667,4.0,1.0,108.0,0.037037,0.009259,0.064516


In [8]:
journey_stats.columns

Index(['journey_id', 'user_pseudo_id', 'tour_id', 'distinct_stories_touched',
       'distinct_stories_completed', 'total_times_completed',
       'max_story_completed_position', 'dropoff_position', 'total_stories',
       'depth_ratio', 'max_depth', 'dropoff', 'total_transitions',
       'sequential_transitions', 'sequential_ratio',
       'completed_sequential_transitions', 'sequential_completed_ratio',
       'distinct_sequential_transitions',
       'distinct_completed_sequential_transitions', 'max_possible_transitions',
       'global_sequential_coverage', 'global_completed_sequential_coverage',
       'interactivity_pct'],
      dtype='str')

### Create the user_stats table

In [9]:
user_stats = (
    journey_stats
    .groupby("user_pseudo_id", as_index=False)
    .agg({
        "journey_id": "count",
        "distinct_stories_touched": "mean",
        "distinct_stories_completed": "mean",
        "total_times_completed": "mean",
        "max_story_completed_position": "mean",
        "dropoff_position": "mean",
        "total_stories": "mean",
        "depth_ratio": "mean",
        "max_depth": "mean",
        "dropoff": "mean",
        "total_transitions": "mean",
        "sequential_transitions": "mean",
        "sequential_ratio": "mean",
        "completed_sequential_transitions": "mean",
        "sequential_completed_ratio": "mean",
        "distinct_sequential_transitions": "mean",
        "distinct_completed_sequential_transitions": "mean",
        "max_possible_transitions": "mean",
        "global_sequential_coverage": "mean",
        "global_completed_sequential_coverage": "mean",
        "interactivity_pct": "mean",
    })
    .rename(columns={"journey_id": "n_journeys"})
)

In [11]:
user_stats["pct_touched"] = (
    user_stats["distinct_stories_touched"] / user_stats["total_stories"]
)

### Users clustering table

In [ ]:
cluster_cols = [
    "depth_ratio",
    "pct_touched",
    "max_depth",
    "dropoff",
    "global_sequential_coverage",
    "global_completed_sequential_coverage",
    "interactivity_pct"
]

In [12]:
user_stats = user_stats.rename(columns={
    "global_sequential_coverage": "sequential_touched",
    "global_completed_sequential_coverage": "sequential_completed"
})

cluster_cols = [
    "depth_ratio",
    "pct_touched",
    "max_depth",
    "dropoff",
    "sequential_touched",
    "sequential_completed",
    "interactivity_pct"
]

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

cluster_cols = [
    "depth_ratio",
    "pct_touched",
    "max_depth",
    "dropoff",
    "sequential_touched",
    "sequential_completed",
    "interactivity_pct"
]

# keep only rows with complete data
cluster_data = user_stats[["user_pseudo_id"] + cluster_cols].dropna().copy()



In [15]:
# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data[cluster_cols])



In [16]:
# test different k values
results = []
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    results.append({"k": k, "silhouette_score": sil})

k_results = pd.DataFrame(results).sort_values("silhouette_score", ascending=False)
k_results

,k,silhouette_score
0,2,0.358604
1,3,0.309932
3,5,0.276092
2,4,0.265065
6,8,0.264388
5,7,0.256843
4,6,0.247665


In [18]:
user_stats_raw = user_stats.copy()

In [23]:
user_stats = user_stats_raw.copy()

In [24]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
cluster_data["cluster"] = kmeans.fit_predict(X_scaled)

# merge cluster labels back to user_stats
user_stats = user_stats.merge(
    cluster_data[["user_pseudo_id", "cluster"]],
    on="user_pseudo_id",
    how="left"
)

In [25]:
cluster_summary = (
    user_stats
    .groupby("cluster")[cluster_cols + ["n_journeys"]]
    .mean()
    .round(3)
)

cluster_summary

,depth_ratio,pct_touched,max_depth,dropoff,sequential_touched,sequential_completed,interactivity_pct,n_journeys
cluster,,,,,,,,
0,0.242,0.408,0.739,0.886,0.242,0.146,0.079,1.370
1,0.627,0.761,0.920,0.961,0.556,0.459,0.068,1.203
2,0.069,0.148,0.212,0.391,0.080,0.040,0.096,1.316


In [28]:
user_stats["cluster"].value_counts()


cluster
0    3718
2    2483
1    1764
Name: count, dtype: int64

In [27]:
cluster_all = (
    user_stats
    .groupby("cluster")
    .mean(numeric_only=True)
    .round(3)
)

cluster_all

,n_journeys,distinct_stories_touched,distinct_stories_completed,total_times_completed,max_story_completed_position,dropoff_position,total_stories,depth_ratio,max_depth,dropoff,...,sequential_ratio,completed_sequential_transitions,sequential_completed_ratio,distinct_sequential_transitions,distinct_completed_sequential_transitions,max_possible_transitions,sequential_touched,sequential_completed,interactivity_pct,pct_touched
cluster,,,,,,,,,,,,,,,,,,,,,
0,1.370,37.912,22.852,25.264,77.050,90.094,100.374,0.242,0.739,0.886,...,0.433,17.109,0.279,23.200,14.534,99.374,0.242,0.146,0.079,0.408
1,1.203,60.609,49.344,54.784,74.682,77.943,80.939,0.627,0.920,0.961,...,0.577,41.450,0.485,43.180,35.523,79.939,0.556,0.459,0.068,0.761
2,1.316,12.737,5.977,6.432,20.802,37.139,90.133,0.069,0.212,0.391,...,0.365,3.987,0.179,6.861,3.492,89.133,0.080,0.040,0.096,0.148


In [29]:
journey_stats = journey_stats.merge(
    cluster_data[["user_pseudo_id", "cluster"]],
    on="user_pseudo_id",
    how="left"
)
journey_stats.head(10)

,journey_id,user_pseudo_id,tour_id,distinct_stories_touched,distinct_stories_completed,total_times_completed,max_story_completed_position,dropoff_position,total_stories,depth_ratio,...,completed_sequential_transitions,sequential_completed_ratio,distinct_sequential_transitions,distinct_completed_sequential_transitions,max_possible_transitions,global_sequential_coverage,global_completed_sequential_coverage,interactivity_pct,pct_touched,cluster
0,12146,c7f26129dd892cb7292f6283113726ad,107,67.0,11.0,12,167.0,220.0,220.0,0.050000,...,8,0.098765,59.0,6.0,219.0,0.269406,0.027397,0.019704,0.304545,0
1,840,0e1e29466bf2de31e00fe7a989673c99,822,2.0,1.0,1,2.0,2.0,82.0,0.012195,...,0,0.000000,0.0,0.0,81.0,0.000000,0.000000,0.000000,0.024390,2
2,841,0e1e29466bf2de31e00fe7a989673c99,822,48.0,23.0,28,81.0,82.0,82.0,0.280488,...,33,0.275000,30.0,15.0,81.0,0.370370,0.185185,0.086364,0.585366,2
3,7295,767fa15a9558b669f47d8e67e719918c,284,41.0,16.0,17,63.0,64.0,67.0,0.238806,...,3,0.027273,13.0,2.0,66.0,0.196970,0.030303,0.076923,0.611940,0
4,10605,ada97a097eabb072201591dd267767cd,447,4.0,0.0,0,0.0,32.0,73.0,0.000000,...,0,0.000000,0.0,0.0,72.0,0.000000,0.000000,0.210526,0.054795,2
5,10457,aa95a709d2859cb0b976835344d7df10,859,17.0,8.0,8,73.0,73.0,113.0,0.070796,...,1,0.050000,3.0,1.0,112.0,0.026786,0.008929,0.015385,0.150442,2
6,10842,b22d1794fe798eea9481aaaee845d122,240,39.0,26.0,26,77.0,79.0,80.0,0.325000,...,5,0.111111,10.0,5.0,79.0,0.126582,0.063291,0.013029,0.487500,0
7,10844,b22d1794fe798eea9481aaaee845d122,278,62.0,38.0,38,104.0,103.0,105.0,0.361905,...,7,0.083333,9.0,7.0,104.0,0.086538,0.067308,0.004717,0.590476,0
8,10845,b22d1794fe798eea9481aaaee845d122,862,11.0,9.0,9,40.0,40.0,42.0,0.214286,...,1,0.066667,1.0,1.0,41.0,0.024390,0.024390,0.000000,0.261905,0
9,9658,9cd63ad90a923a99be3d27a1e6085bd0,644,19.0,7.0,7,50.0,49.0,109.0,0.064220,...,2,0.066667,4.0,1.0,108.0,0.037037,0.009259,0.064516,0.174312,2


In [30]:
journey_stats.to_csv(BASE_DIR / 'data/clean/journey_clusters.csv')